# Did DRS Increase Overtaking?: exploration

Scratch notebook for the analysis in `src/`. Run `src/ingest.py`, `src/build_drs_zones.py`,
`src/overtakes.py` and `src/panel.py` first so the parquet files exist.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd, numpy as np
from panel import build_panel, BREAK_YEAR
from its import naive_its, dose_response, placebos, robustness, fmt
from power import mde_closed_form, residual_treatment_variation

p = build_panel()
p.shape

## 1. The sanity check that has to pass first

Overtakes per race by season. If there is no visible jump at 2011, something upstream is broken.

In [ ]:
season = p.groupby("year")[["overtakes_strict", "overtakes_loose"]].mean().round(1)
season

In [ ]:
print("pre-2011 mean :", p[p.year < BREAK_YEAR].overtakes_strict.mean().round(2))
print("post-2011 mean:", p[p.year >= BREAK_YEAR].overtakes_strict.mean().round(2))

## 2. Treatment intensity

Zone count per circuit-season, and how much of it survives the fixed effects.

In [ ]:
post = p[(p.year >= BREAK_YEAR) & p.n_drs_zones.notna()]
print(post.n_drs_zones.value_counts().sort_index())
ch = post.groupby("circuitRef").n_drs_zones.nunique()
print(f"\ncircuits that changed zone count: {(ch > 1).sum()} of {len(ch)}")
residual_treatment_variation(p)

## 3. The naive ITS: the model this project argues against

In [ ]:
res, terms = naive_its(p)
for k, v in terms.items():
    print(f"{k:10s} {fmt(v)}")

## 4. The dose-response panel: the real estimate

`C(year)` absorbs the Pirelli tyre change, which hit every circuit equally.

In [ ]:
res, d = dose_response(p)
print(fmt(d, "overtakes/race/zone"))
print("MDE at 80% power:", round(mde_closed_form(d["se"]), 2))

In [ ]:
robustness(p)[["check", "coef", "ci_low", "ci_high", "p", "n"]].round(3)

## 5. Placebo break dates (pre-treatment period only)

In [ ]:
placebos(p).round(3)

## 6. What would DRS need to be, to explain the whole 2011 jump?

In [ ]:
shift = terms["post"]["coef"]
mean_zones = post.n_drs_zones.mean()
needed = shift / mean_zones
print(f"level shift        : {shift:+.2f}")
print(f"mean zones         : {mean_zones:.2f}")
print(f"needed per zone    : {needed:+.2f}")
print(f"our CI upper bound : {d['ci_high']:+.2f}")
print("ruled out at 5%   :", needed > d["ci_high"])